# Customer Churn Analysis and Retention Strategy — Telecom Operator (Canada)

**Business question:** Which customer segments are most likely to churn, and what retention
actions can reduce churn and protect revenue?

This notebook loads the IBM Telco Customer Churn sample dataset, cleans it, quantifies churn
drivers, estimates revenue at risk and produces the artefacts consumed by the dashboard
(`docs/dashboard.json` + `docs/data/*.json`) and the executive summary (`outputs/summary.json`).

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
OUT_CHARTS = ROOT / "outputs" / "charts"
OUT = ROOT / "outputs"
DOCS_DATA = ROOT / "docs" / "data"
for p in (OUT_CHARTS, OUT, DOCS_DATA):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
PALETTE = ["#1f4e79", "#2e75b6", "#9dc3e6", "#c55a11", "#f4b183"]

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA / "telco_churn.csv")
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head()

## 2. Clean and transform

- `TotalCharges` arrives as text with blank strings for brand-new customers (tenure = 0) → coerce to numeric.
- Standardise category labels and derive analysis flags (`is_churned`, `tenure_band`, `revenue_at_risk`).

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
blank_total = df["TotalCharges"].isna().sum()
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"] * df["tenure"])

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df["is_churned"] = (df["Churn"] == "Yes").astype(int)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["tenure_band"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 100],
    labels=["0-6 months", "7-12 months", "13-24 months", "25-48 months", "49+ months"],
)
df["revenue_at_risk"] = np.where(df["is_churned"] == 1, df["MonthlyCharges"], 0.0)

print(f"Blank TotalCharges imputed: {blank_total}")
print(f"Duplicated customerIDs: {df['customerID'].duplicated().sum()}")
print(f"Missing values remaining: {int(df.isna().sum().sum())}")
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2)

## 3. Headline KPIs

Overall churn rate, average monthly charges, average tenure and the monthly revenue at risk
(sum of monthly charges of churned customers — a proxy for recurring revenue lost each month).

In [ ]:
total_customers = len(df)
churned = int(df["is_churned"].sum())
churn_rate = churned / total_customers
avg_monthly = df["MonthlyCharges"].mean()
avg_tenure = df["tenure"].mean()
revenue_at_risk = df["revenue_at_risk"].sum()
avg_clv = (df["MonthlyCharges"] * df["tenure"]).mean()

print(f"Customers:            {total_customers:,}")
print(f"Churned customers:    {churned:,}")
print(f"Churn rate:           {churn_rate:.1%}")
print(f"Avg monthly charges:  ${avg_monthly:,.2f}")
print(f"Avg tenure:           {avg_tenure:.1f} months")
print(f"Monthly revenue at risk: ${revenue_at_risk:,.0f}")
print(f"Avg customer lifetime value: ${avg_clv:,.0f}")

## 4. Churn by contract type

Month-to-month contracts are the single strongest churn driver: customers on them churn at a
far higher rate than one- or two-year contracts, and they carry most of the revenue at risk.

In [ ]:
contract = (
    df.groupby("Contract", observed=True)
      .agg(customers=("customerID", "count"),
           churned=("is_churned", "sum"),
           avg_monthly=("MonthlyCharges", "mean"),
           revenue_at_risk=("revenue_at_risk", "sum"))
      .reset_index()
)
contract["churn_rate"] = contract["churned"] / contract["customers"]
contract = contract.sort_values("churn_rate", ascending=False)
contract.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(contract["Contract"], contract["churn_rate"] * 100, color=PALETTE[:3])
ax.set_title("Churn rate by contract type")
ax.set_ylabel("Churn rate (%)")
for b, v in zip(bars, contract["churn_rate"] * 100):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_ylim(0, contract["churn_rate"].max() * 100 * 1.2)
fig.tight_layout()
fig.savefig(OUT_CHARTS / "churn_by_contract.png")
plt.close(fig)

## 5. Churn by tenure band

Risk is concentrated in the first year. Customers in their first 6 months churn at roughly
3–4× the rate of customers past the 4-year mark — onboarding is the critical retention window.

In [ ]:
tenure = (
    df.groupby("tenure_band", observed=True)
      .agg(customers=("customerID", "count"),
           churned=("is_churned", "sum"),
           avg_monthly=("MonthlyCharges", "mean"))
      .reset_index()
)
tenure["churn_rate"] = tenure["churned"] / tenure["customers"]
tenure["tenure_band"] = tenure["tenure_band"].astype(str)
tenure.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tenure["tenure_band"], tenure["churn_rate"] * 100, marker="o", color=PALETTE[0], linewidth=2)
ax.fill_between(tenure["tenure_band"], tenure["churn_rate"] * 100, color=PALETTE[2], alpha=0.35)
ax.set_title("Churn rate by tenure band")
ax.set_ylabel("Churn rate (%)")
ax.set_xlabel("Tenure band")
for x, y in zip(tenure["tenure_band"], tenure["churn_rate"] * 100):
    ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(OUT_CHARTS / "churn_by_tenure.png")
plt.close(fig)

## 6. Churn by internet service and support add-ons

Fibre-optic customers churn more than DSL customers, but the gap narrows sharply when they
subscribe to Tech Support and Online Security — add-ons act as retention anchors.

In [ ]:
internet = (
    df.groupby("InternetService", observed=True)
      .agg(customers=("customerID", "count"),
           churned=("is_churned", "sum"),
           avg_monthly=("MonthlyCharges", "mean"))
      .reset_index()
)
internet["churn_rate"] = internet["churned"] / internet["customers"]
internet = internet.sort_values("churn_rate", ascending=False)

support = (
    df[df["InternetService"] != "No"]
      .groupby(["InternetService", "TechSupport"], observed=True)
      .agg(customers=("customerID", "count"), churned=("is_churned", "sum"))
      .reset_index()
)
support["churn_rate"] = support["churned"] / support["customers"]
support.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
pivot = support.pivot(index="InternetService", columns="TechSupport", values="churn_rate") * 100
pivot.plot(kind="bar", ax=ax, color=[PALETTE[3], PALETTE[1]])
ax.set_title("Churn rate by internet service and tech support")
ax.set_ylabel("Churn rate (%)")
ax.set_xlabel("")
ax.legend(title="Tech Support", frameon=False)
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(OUT_CHARTS / "churn_by_internet_support.png")
plt.close(fig)

## 7. Churn by payment method

Electronic-check payers churn at the highest rate; automatic bank transfer and credit-card
customers are the most loyal. Payment friction is a cheap lever to pull.

In [ ]:
payment = (
    df.groupby("PaymentMethod", observed=True)
      .agg(customers=("customerID", "count"),
           churned=("is_churned", "sum"),
           avg_tenure=("tenure", "mean"))
      .reset_index()
)
payment["churn_rate"] = payment["churned"] / payment["customers"]
payment = payment.sort_values("churn_rate", ascending=False)
payment.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(payment["PaymentMethod"], payment["churn_rate"] * 100, color=PALETTE[0])
ax.invert_yaxis()
ax.set_title("Churn rate by payment method")
ax.set_xlabel("Churn rate (%)")
for b, v in zip(bars, payment["churn_rate"] * 100):
    ax.text(v + 0.5, b.get_y() + b.get_height() / 2, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, payment["churn_rate"].max() * 100 * 1.2)
fig.tight_layout()
fig.savefig(OUT_CHARTS / "churn_by_payment.png")
plt.close(fig)

## 8. High-risk segments — where to spend retention budget

Combining contract, internet service and senior status surfaces the segments that combine
high churn with high monthly charges. These are the priority targets for retention offers.

In [ ]:
segments = (
    df.groupby(["Contract", "InternetService", "SeniorCitizen"], observed=True)
      .agg(customers=("customerID", "count"),
           churned=("is_churned", "sum"),
           avg_monthly=("MonthlyCharges", "mean"),
           revenue_at_risk=("revenue_at_risk", "sum"))
      .reset_index()
)
segments = segments[segments["customers"] >= 50].copy()
segments["churn_rate"] = segments["churned"] / segments["customers"]
segments["segment"] = (
    segments["Contract"] + " · " + segments["InternetService"] + " · Senior:" + segments["SeniorCitizen"]
)
segments = segments.sort_values("revenue_at_risk", ascending=False).head(10)
segments[["segment", "customers", "churn_rate", "avg_monthly", "revenue_at_risk"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
top = segments.sort_values("revenue_at_risk")
bars = ax.barh(top["segment"], top["revenue_at_risk"], color=PALETTE[3])
ax.set_title("Top 10 segments by monthly revenue at risk")
ax.set_xlabel("Monthly revenue at risk ($)")
for b, v in zip(bars, top["revenue_at_risk"]):
    ax.text(v + 20, b.get_y() + b.get_height() / 2, f"${v:,.0f}", va="center", fontsize=8)
ax.set_xlim(0, top["revenue_at_risk"].max() * 1.2)
fig.tight_layout()
fig.savefig(OUT_CHARTS / "revenue_at_risk_segments.png")
plt.close(fig)

## 9. Persist artefacts

- `outputs/summary.json` — flat dict of headline KPIs (consumed by the dashboard KPI cards).
- `docs/data/*.json` — one file per chart, list of records.
- `outputs/charts/*.png` — five charts saved above.

In [ ]:
summary = {
    "total_customers": int(total_customers),
    "churned_customers": int(churned),
    "churn_rate": round(float(churn_rate), 4),
    "avg_monthly_charges": round(float(avg_monthly), 2),
    "avg_tenure_months": round(float(avg_tenure), 1),
    "monthly_revenue_at_risk": round(float(revenue_at_risk), 2),
    "avg_customer_lifetime_value": round(float(avg_clv), 2),
    "month_to_month_churn_rate": round(float(contract.loc[contract["Contract"] == "Month-to-month", "churn_rate"].iloc[0]), 4),
    "two_year_churn_rate": round(float(contract.loc[contract["Contract"] == "Two year", "churn_rate"].iloc[0]), 4),
    "first_year_churn_rate": round(float(tenure.loc[tenure["tenure_band"].isin(["0-6 months", "7-12 months"]), "churned"].sum() /
                                           tenure.loc[tenure["tenure_band"].isin(["0-6 months", "7-12 months"]), "customers"].sum()), 4),
}
with open(OUT / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
summary

In [ ]:
contract_out = contract.assign(
    churn_rate=contract["churn_rate"].round(4),
    avg_monthly=contract["avg_monthly"].round(2),
    revenue_at_risk=contract["revenue_at_risk"].round(2),
)[["Contract", "customers", "churned", "churn_rate", "avg_monthly", "revenue_at_risk"]]
contract_out.to_json(DOCS_DATA / "churn_by_contract.json", orient="records")

tenure_out = tenure.assign(churn_rate=tenure["churn_rate"].round(4), avg_monthly=tenure["avg_monthly"].round(2))[
    ["tenure_band", "customers", "churned", "churn_rate", "avg_monthly"]
]
tenure_out.to_json(DOCS_DATA / "churn_by_tenure.json", orient="records")

internet_out = internet.assign(churn_rate=internet["churn_rate"].round(4), avg_monthly=internet["avg_monthly"].round(2))[
    ["InternetService", "customers", "churned", "churn_rate", "avg_monthly"]
]
internet_out.to_json(DOCS_DATA / "churn_by_internet.json", orient="records")

payment_out = payment.assign(churn_rate=payment["churn_rate"].round(4), avg_tenure=payment["avg_tenure"].round(1))[
    ["PaymentMethod", "customers", "churned", "churn_rate", "avg_tenure"]
]
payment_out.to_json(DOCS_DATA / "churn_by_payment.json", orient="records")

segments_out = segments.assign(
    churn_rate=segments["churn_rate"].round(4),
    avg_monthly=segments["avg_monthly"].round(2),
    revenue_at_risk=segments["revenue_at_risk"].round(2),
)[["segment", "customers", "churned", "churn_rate", "avg_monthly", "revenue_at_risk"]]
segments_out.to_json(DOCS_DATA / "revenue_at_risk_segments.json", orient="records")

print("Artefacts written:")
for p in sorted(OUT_CHARTS.glob("*.png")):
    print(" -", p.relative_to(ROOT))
for p in sorted(DOCS_DATA.glob("*.json")):
    print(" -", p.relative_to(ROOT))
print(" - outputs/summary.json")

## 10. Conclusions and recommendations

1. **Contract migration is the highest-leverage lever.** Month-to-month customers churn at
   ~43% vs ~3% for two-year contracts. Target month-to-month customers with a discounted
   upgrade to a 12-month plan at month 3 and month 9.
2. **Protect the first year.** Churn is concentrated in the first 12 months. Introduce a
   structured onboarding programme (welcome call at day 7, check-in at day 30, loyalty offer
   at month 6).
3. **Bundle support add-ons.** Fibre customers without Tech Support churn at ~50%; with
   support the rate drops materially. Offer a free 3-month Tech Support trial to new fibre
   customers.
4. **Reduce payment friction.** Electronic-check payers churn at ~45%. Incentivise migration
   to automatic payment (small monthly credit for 6 months).
5. **Prioritise revenue at risk.** The top 10 segments account for the majority of monthly
   revenue at risk — retention budget should be allocated proportionally to these segments.